# APBS electrostatics: ligand only

Amber-parameterized APBS electrostatic potential map for a single
small-molecule ligand (`ligand.pdb`, residue `l01`).

Pipeline:

1. Assign ligand bond orders from a SMILES template and write an SDF for
   antechamber (PDB files lack bond-order information).
2. Parameterize the ligand with AmberTools/GAFF2 (`obabel`, `antechamber`
   AM1-BCC, `parmchk2`, `tleap`, ParmEd PQR export).
3. Solve APBS on the ligand PQR to produce `ligand.dx`.

Activate the AmberTools environment before launching Jupyter:

```bash
conda activate ambertools
```

In [ ]:
import os
import shutil
from pathlib import Path

import parmed as pmd
from rdkit import Chem

from mdpp.prep import assign_topology, write_apbs_input

# Input (this directory).
EXAMPLE_DIR = Path.cwd()
LIGAND_PDB = EXAMPLE_DIR / "ligand.pdb"

# Workspace layout: each stage owns a folder under tmp/, with transient
# tool files kept in <stage>/intermediate/.
TMP_ROOT = EXAMPLE_DIR / "tmp"
PREP_DIR = TMP_ROOT / "prep"
PREP_INTERMEDIATE = PREP_DIR / "intermediate"
AMBERTOOLS_DIR = TMP_ROOT / "ambertools"
AMBERTOOLS_INTERMEDIATE = AMBERTOOLS_DIR / "intermediate"
APBS_DIR = TMP_ROOT / "apbs"
APBS_INTERMEDIATE = APBS_DIR / "intermediate"
for path in (
    PREP_DIR,
    PREP_INTERMEDIATE,
    AMBERTOOLS_DIR,
    AMBERTOOLS_INTERMEDIATE,
    APBS_DIR,
    APBS_INTERMEDIATE,
):
    path.mkdir(parents=True, exist_ok=True)

# Ligand template (assigns bond orders to the heavy-atom-only PDB ligand).
LIGAND_SMILES = (
    r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)"
    r"[C@@H](O)[C@H]3O"
)
LIG_RESNAME = "l01"

# AmberTools (ligand parameterization).
LIGAND_FF = "leaprc.gaff2"
PB_RADII = "mbondi3"

# Export every shell-facing knob so the %%bash cells inherit the same
# configuration without re-typing defaults.
os.environ.update({
    "AMBERTOOLS_DIR": str(AMBERTOOLS_DIR),
    "AMBERTOOLS_INTERMEDIATE": str(AMBERTOOLS_INTERMEDIATE),
    "APBS_DIR": str(APBS_DIR),
    "APBS_INTERMEDIATE": str(APBS_INTERMEDIATE),
    "LIGAND_FF": LIGAND_FF,
    "PB_RADII": PB_RADII,
    "LIG_RESNAME": LIG_RESNAME,
})

## Step 1: Assign ligand topology

Assign bond orders from the SMILES template to the ligand heavy-atom
coordinates, add hydrogens, and write an SDF for antechamber. The net charge
read from the template is passed to antechamber.

In [ ]:
# Assign bond orders from the SMILES template.
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(LIGAND_PDB), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)
mol_assigned.SetProp("_Name", LIG_RESNAME)

ligand_sdf = PREP_DIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)

(PREP_DIR / "ligand_charge.txt").write_text(f"{ligand_net_charge}\n")
(PREP_DIR / "ligand_resname.txt").write_text(f"{LIG_RESNAME}\n")

# Make the net charge available to the antechamber bash cell.
os.environ["NET_CHARGE"] = str(ligand_net_charge)
print(f"Ligand SDF -> {ligand_sdf}")

## Step 2: Parameterize the ligand with AmberTools

1. `obabel` converts the ligand SDF to mol2 as an antechamber seed; we then
   patch the residue name to match the ligand.
2. `antechamber` assigns AM1-BCC charges and GAFF2 atom types.
3. `parmchk2` generates any missing GAFF2 parameters.
4. `tleap` loads GAFF2 and saves the prmtop/rst7 topology.
5. ParmEd exports the topology to PQR (charge + radius per atom).

In [ ]:
# Stage the ligand SDF into the AmberTools intermediate folder.
shutil.copy(ligand_sdf, AMBERTOOLS_INTERMEDIATE / "ligand.sdf")

In [ ]:
%%bash
set -euo pipefail
cd "$AMBERTOOLS_INTERMEDIATE"

echo "=== 1. obabel: SDF -> mol2 ==="
obabel ligand.sdf -O ligand_seed.mol2

In [ ]:
# Patch the residue name written by obabel so it matches the ligand.
ligand_seed_mol2 = AMBERTOOLS_INTERMEDIATE / "ligand_seed.mol2"
text = ligand_seed_mol2.read_text()
for old in ("UNL1", "UNL", "UNK"):
    text = text.replace(old, LIG_RESNAME)
if LIG_RESNAME not in text:
    raise RuntimeError(
        f"Residue name {LIG_RESNAME!r} not found in {ligand_seed_mol2} after replacement"
    )
ligand_seed_mol2.write_text(text)
print(f"Patched residue name -> {LIG_RESNAME}")

In [ ]:
%%bash
set -euo pipefail
cd "$AMBERTOOLS_INTERMEDIATE"

echo "=== 2. antechamber (AM1-BCC + GAFF2) ==="
antechamber \
    -i ligand_seed.mol2 -fi mol2 \
    -o ligand_amber.mol2 -fo mol2 \
    -c bcc -s 2 -at gaff2 \
    -nc "$NET_CHARGE" -rn "$LIG_RESNAME"

echo "=== 3. parmchk2 (missing GAFF2 parameters) ==="
parmchk2 -i ligand_amber.mol2 -f mol2 -o ligand.frcmod

echo "=== 4. tleap (GAFF2 ligand) ==="
cat >tleap.in <<EOF
source $LIGAND_FF

$LIG_RESNAME = loadmol2 ligand_amber.mol2
loadamberparams ligand.frcmod

set default PBRadii $PB_RADII
saveamberparm $LIG_RESNAME ligand.prmtop ligand.rst7
quit
EOF
tleap -f tleap.in

In [ ]:
# ParmEd: convert each topology/coord pair to PQR (charge + radius per atom).
for stem in ("ligand",):
    parm = pmd.load_file(
        str(AMBERTOOLS_INTERMEDIATE / f"{stem}.prmtop"),
        xyz=str(AMBERTOOLS_INTERMEDIATE / f"{stem}.rst7"),
    )
    parm.save(str(AMBERTOOLS_INTERMEDIATE / f"{stem}.pqr"), overwrite=True)
    for suffix in ("prmtop", "rst7", "pqr"):
        shutil.copy(
            AMBERTOOLS_INTERMEDIATE / f"{stem}.{suffix}",
            AMBERTOOLS_DIR / f"{stem}.{suffix}",
        )

print("AmberTools PQR outputs:")
for stem in ("ligand",):
    print("  ", AMBERTOOLS_DIR / f"{stem}.pqr")

## Step 3: Solve APBS for the ligand

Generate `ligand.in` for the ligand PQR and run `apbs` to produce `ligand.dx`.

In [ ]:
# Stage ligand.pqr into the APBS intermediate folder and write ligand.in.
shutil.copy(AMBERTOOLS_DIR / "ligand.pqr", APBS_INTERMEDIATE / "ligand.pqr")
apbs_in = write_apbs_input("ligand", APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

In [ ]:
%%bash
set -euo pipefail
cd "$APBS_INTERMEDIATE"

stem=ligand
echo "=== APBS: $stem ==="
apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

if [[ -s "$stem-PE0.dx" ]]; then
    mv "$stem-PE0.dx" "$stem.dx"
elif [[ -s "$stem.pqr.dx" ]]; then
    mv "$stem.pqr.dx" "$stem.dx"
fi

if [[ ! -s "$stem.dx" ]]; then
    echo "Missing $stem.dx" >&2
    exit 1
fi
cp "$stem.in" "$APBS_DIR/$stem.in"
cp "$stem.apbs.log" "$APBS_DIR/$stem.apbs.log"
cp "$stem.dx" "$APBS_DIR/$stem.dx"
ls -lh "$APBS_DIR/$stem.dx"